# Module 2: Hooks (10 min)

Add a rate limiter hook to the customer service agent. Hooks let you inject deterministic code into the agent loop — before invocations, before/after tool calls — without changing the agent's logic.

**Prerequisites:** Module 1 completed, `strands-agents` installed

In [5]:
!pip install -q -r requirements.txt

In [6]:
# AWS-sponsored events / AWS credits
# If you are running this workshop with AWS-provided credits, those credits
# only work with Amazon Nova models — not Claude (the default).
# To switch, import BedrockModel and pass it to Agent(...):
#
# from strands.models import BedrockModel
# model = BedrockModel(model_id="amazon.nova-pro-v1:0")
# agent = Agent(model=model, tools=[...], hooks=[...], system_prompt=...)
#
# Available Nova model IDs: https://docs.aws.amazon.com/bedrock/latest/userguide/model-cards-amazon.html
#   amazon.nova-micro-v1:0  — fastest, text-only, lowest cost
#   amazon.nova-lite-v1:0   — low-cost, multimodal (text, image, video)
#   amazon.nova-pro-v1:0    — balanced accuracy/speed, multimodal (recommended)
#
# Run locally without AWS credentials using Ollama (https://ollama.com/download):
#   1. Install Ollama — macOS: DMG at ollama.com | Linux: curl -fsSL https://ollama.com/install.sh | sh
#   2. ollama pull llama3.1   (supports tool use)
#   3. pip install strands-agents[ollama]
#   4. from strands.models import OllamaModel
#      model = OllamaModel(host="http://localhost:11434", model_id="llama3.1")
#      agent = Agent(model=model, tools=[...], hooks=[...], system_prompt=...)
#   Other models with tool support: llama3.2, qwen2.5, qwen3, mistral

---

## Part 1: Build a Rate Limiter Hook

Hooks implement `HookProvider` and register callbacks for lifecycle events. This one caps each tool at 3 calls per request.

In [7]:
from strands import Agent, tool
from strands.hooks import (
    HookProvider, HookRegistry,
    BeforeInvocationEvent, BeforeToolCallEvent,
)
from customer_service_tools import lookup_customer, get_order_history, process_refund


class RateLimiterHook(HookProvider):
    """Caps each tool at max_calls per agent invocation."""

    def __init__(self, max_calls: int = 3):
        self.max_calls = max_calls
        self.counts: dict[str, int] = {}

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeInvocationEvent, self.reset)
        registry.add_callback(BeforeToolCallEvent, self.check)

    def reset(self, event: BeforeInvocationEvent) -> None:
        """Reset counts at the start of each invocation."""
        self.counts = {}
        print("[HOOK] 🔄 Rate limiter reset")

    def check(self, event: BeforeToolCallEvent) -> None:
        """Check and enforce the rate limit before each tool call."""
        name = event.tool_use["name"]
        self.counts[name] = self.counts.get(name, 0) + 1
        print(f"[HOOK] 📊 {name}: call {self.counts[name]}/{self.max_calls}")

        if self.counts[name] > self.max_calls:
            event.cancel_tool = (
                f"'{name}' hit the {self.max_calls}-call limit. "
                "Do NOT call this tool again."
            )
            print(f"[HOOK] 🚫 BLOCKED: {name} exceeded limit!")


print("✅ RateLimiterHook defined")

✅ RateLimiterHook defined


---

## Part 2: Attach the Hook to the Agent

Pass hooks via the `hooks` parameter. The agent loop will call your hook at the right lifecycle points.

In [8]:
SYSTEM_PROMPT = """You are a customer service agent for an online electronics store.
Be helpful, professional, and concise. Use the available tools to look up customer
information and process requests."""

agent = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    hooks=[RateLimiterHook(max_calls=3)],
    system_prompt=SYSTEM_PROMPT,
)

# Normal request — should work fine
result = agent("I'm customer C-1001. What are my recent orders?")

[HOOK] 🔄 Rate limiter reset


Let me look up your account and order history at the same time!
Tool #1: lookup_customer

Tool #2: get_order_history
[HOOK] 📊 lookup_customer: call 1/3
[HOOK] 📊 get_order_history: call 1/3
Hi Sarah! Here's a summary of your recent orders:

| Order ID | Item | Price | Status | Date |
|----------|------|-------|--------|------|
| ORD-5521 | Wireless Headphones | $79.99 | ✅ Delivered | Ordered Apr 20, delivered Apr 28 |
| ORD-5488 | USB-C Hub | $45.00 | 🚚 Shipped | Ordered May 1, est. delivery May 6 |

Is there anything else you'd like help with, such as a return or any issues with these orders?

---

## Part 3: Trigger the Rate Limit

Ask the agent something that would cause excessive tool calls. Watch the hook block it.

In [9]:
# This prompt tries to get the agent to call lookup_customer many times
agent = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    hooks=[RateLimiterHook(max_calls=2)],  # Lower limit to trigger faster
    system_prompt=SYSTEM_PROMPT,
)

result = agent(
    "Look up customers C-1001, C-1002, C-1003, C-1004, and C-1005. "
    "Give me all their details."
)

[HOOK] 🔄 Rate limiter reset
I'll look up all five customers at the same time right away!
Tool #1: lookup_customer

Tool #2: lookup_customer

Tool #3: lookup_customer

Tool #4: lookup_customer

Tool #5: lookup_customer
[HOOK] 📊 lookup_customer: call 1/2
[HOOK] 📊 lookup_customer: call 2/2
[HOOK] 📊 lookup_customer: call 3/2
[HOOK] 🚫 BLOCKED: lookup_customer exceeded limit!
[HOOK] 📊 lookup_customer: call 4/2
[HOOK] 🚫 BLOCKED: lookup_customer exceeded limit!
[HOOK] 📊 lookup_customer: call 5/2
[HOOK] 🚫 BLOCKED: lookup_customer exceeded limit!
Here are the details I was able to retrieve. Unfortunately, I ran into a system limitation that prevented me from pulling all five records:

---

### ✅ Successfully Retrieved

| Customer ID | Name | Email | Phone | Status |
|---|---|---|---|---|
| **C-1001** | Sarah Johnson | sarah.johnson@email.com | 555-0142 | 🟢 Active |
| **C-1002** | Mike Chen | mike.chen@email.com | 555-0198 | 🟢 Active |

---

### ❌ Unable to Retrieve

- **C-1003**, **C-1004**, and

---

## 🎯 Try It Yourself

Modify the hook to also track total calls across all tools (not just per-tool):

In [13]:
# Challenge: Add a total_calls counter that blocks ALL tools after 5 total calls
# Hint: Add a self.total counter in reset() and check it in check()

from strands import Agent, tool
from strands.hooks import (
    HookProvider,
    HookRegistry,
    BeforeInvocationEvent,
    BeforeToolCallEvent,
)
from customer_service_tools import (
    lookup_customer,
    get_order_history,
    process_refund,
)


# Part 1: Define Rate Limiter Hook

class RateLimiterHook(HookProvider):
    """Limits individual tool calls and total tool calls per invocation."""

    def __init__(self, max_calls: int = 3, max_total_calls: int = 5):
        self.max_calls = max_calls
        self.max_total_calls = max_total_calls
        self.counts: dict[str, int] = {}
        self.total_calls = 0

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeInvocationEvent, self.reset)
        registry.add_callback(BeforeToolCallEvent, self.check)

    def reset(self, event: BeforeInvocationEvent) -> None:
        """Reset counters at the start of each invocation."""
        self.counts = {}
        self.total_calls = 0
        print("\n[HOOK] Rate limiter reset")

    def check(self, event: BeforeToolCallEvent) -> None:
        """Check per-tool and global limits before tool execution."""
        name = event.tool_use["name"]

        self.counts[name] = self.counts.get(name, 0) + 1
        self.total_calls += 1

        print(
            f"[HOOK] {name}: "
            f"{self.counts[name]}/{self.max_calls}"
        )
        print(
            f"[HOOK] Total calls: "
            f"{self.total_calls}/{self.max_total_calls}"
        )

        if self.total_calls > self.max_total_calls:
            event.cancel_tool = (
                f"Global limit of {self.max_total_calls} tool calls exceeded. "
                "Do not call any more tools."
            )
            print("[HOOK] BLOCKED: Global tool-call limit exceeded!")

        elif self.counts[name] > self.max_calls:
            event.cancel_tool = (
                f"Per-tool limit of {self.max_calls} calls exceeded for {name}."
            )
            print(f"[HOOK] BLOCKED: {name} exceeded its limit!")


print("RateLimiterHook defined successfully!")


# Part 2: Configure the Agent

SYSTEM_PROMPT = """
You are a customer service agent for an online electronics store.

Be helpful, professional, and concise.

Use available tools to look up customer information and process requests.
Always verify the customer before taking action.
Do not repeatedly call tools when a limit is reached.
"""

rate_limiter = RateLimiterHook(
    max_calls=3,
    max_total_calls=5
)

agent = Agent(
    tools=[
        lookup_customer,
        get_order_history,
        process_refund
    ],
    hooks=[rate_limiter],
    system_prompt=SYSTEM_PROMPT,
)

print("Agent initialized successfully!")


# Part 3: Test the Rate Limiter

print("\n--- Testing Global Rate Limiter ---")

result = agent(
    "Look up customers C-1001, C-1002, C-1003, "
    "C-1004, C-1005, and C-1006. "
    "Give me their details."
)

print("\n--- Agent Final Response ---")
print(result)

print("\n--- Rate Limiter Statistics ---")
print("Tool call counts:", rate_limiter.counts)
print("Total tool calls:", rate_limiter.total_calls)

print("\nModule 2 challenge completed!")


RateLimiterHook defined successfully!
Agent initialized successfully!

--- Testing Global Rate Limiter ---

[HOOK] Rate limiter reset
I'll look up all six customers simultaneously right away!
Tool #1: lookup_customer

Tool #2: lookup_customer

Tool #3: lookup_customer

Tool #4: lookup_customer

Tool #5: lookup_customer

Tool #6: lookup_customer
[HOOK] lookup_customer: 1/3
[HOOK] Total calls: 1/5
[HOOK] lookup_customer: 2/3
[HOOK] Total calls: 2/5
[HOOK] lookup_customer: 3/3
[HOOK] Total calls: 3/5
[HOOK] lookup_customer: 4/3
[HOOK] Total calls: 4/5
[HOOK] BLOCKED: lookup_customer exceeded its limit!
[HOOK] lookup_customer: 5/3
[HOOK] Total calls: 5/5
[HOOK] BLOCKED: lookup_customer exceeded its limit!
[HOOK] lookup_customer: 6/3
[HOOK] Total calls: 6/5
[HOOK] BLOCKED: Global tool-call limit exceeded!
Here's a summary of the results for all six customers:

| Customer ID | Name | Email | Phone | Status |
|-------------|------|-------|-------|--------|
| **C-1001** | Sarah Johnson | sarah

---

## 💬 Want a real multi-turn conversation?

In a notebook, each cell is a **single turn**. To chat back and forth with the rate-limited agent, run the companion script in a **terminal**, which wraps this same agent (with `RateLimiterHook`) in a loop. From the cloned repo:

```bash
cd samples/02-hooks
pip install -r requirements.txt
python chat.py
```

Type your messages, and `quit` (or Ctrl+C) to exit. The rate limiter resets at the start of each turn, and the agent keeps its context across turns.

---

## What's Next

Hooks give you low-level control over the loop. In **Module 3: Skills + Steering**, you'll add workflow knowledge (skills) and business rule enforcement (steering handlers) to make the agent follow proper procedures.